# 14.5 Named problems

In [1]:
# install dependencies
%pip install -q amplpy

from amplpy import AMPL, ampl_notebook

ampl = ampl_notebook(
    modules=['highs'],  # modules to install
    license_uuid='default',  # license to use
)  # instantiate AMPL object and register magics

As our cutting-stock example has shown, the key to writing a clear and efficient script
for alternating between two (or more) models lies in working with named problems that

```ampl
ampl: commands cut.run;
CPLEX 8.0.0: optimal solution; objective 52.1
0 dual simplex iterations (0 in phase I)
CPLEX 8.0.0: optimal integer solution; objective  -0.2
1 MIP simplex iterations
0 branch-and-bound nodes
CPLEX 8.0.0: optimal solution; objective 48.6
2 dual simplex iterations (0 in phase I)
CPLEX 8.0.0: optimal integer solution; objective  -0.2
2 MIP simplex iterations
0 branch-and-bound nodes
CPLEX 8.0.0: optimal solution; objective 47
1 dual simplex iterations (0 in phase I)
CPLEX 8.0.0: optimal integer solution; objective   -0.1
2 MIP simplex iterations
0 branch-and-bound nodes
CPLEX 8.0.0: optimal solution; objective 46.25
2 dual simplex iterations (0 in phase I)
CPLEX 8.0.0: optimal integer solution; objective  -1e-06
8 MIP simplex iterations
0 branch-and-bound nodes
nbr [*,*]
:    1   2    3    4    5    6    7    8 :=
20   5   0    0    0    0    1    1    3
45   0   2    0    0    0    2    0    0
50   0   0    2    0    0    0    0    1
55   0   0    0    2    0    0    0    0
75   0   0    0    0    1    0    1    0
;
Cut [*] :=
1 0  2 0  3  8.25  4  5  5  0  6 17.5  7  8  8  7.5
;
CPLEX 8.0.0: optimal integer solution; objective 47
5 MIP simplex iterations
0 branch-and-bound nodes
Cut [*] :=
1 0    2 0   3   8   4   5   5  0   6 18   7   8   8   8
;
```

**Figure 14-5:** Output from execution of Figure 14-3 cutting-stock script.

represent different subsets of `model` components. In this section we describe in more
detail how AMPL's problem statement is employed to define, use, and `display` named
problems. At the end we also introduce a similar idea, named environments, which facilitates
switching between collections of AMPL options.
Illustrations in this section are taken from the cutting-stock script and from some of
the other example scripts on the AMPL web site. An explanation of the logic behind these
scripts is beyond the scope of this book; some suggestions for learning more are given in
the references at the end of the chapter.

**Defining named problems**

At any point during an AMPL session, there is a current problem consisting of a list of
variables, objectives and constraints. The current problem is named Initial by
default, and comprises all variables, objectives and constraints defined so far. You can
define other "named" problems consisting of subsets of these components, however, and
can make them current. When a named problem is made current, all of the `model` components
in the problem's subset are made active, while all other variables, objectives and
constraints are made inactive. More precisely, variables in the problem's subset are
unfixed and the remainder are fixed at their current values. Objectives and constraints in
the problem's subset are restored and the remainder are dropped. (Fixing and dropping
are discussed in [Section 11.4](../11/11_4_modifying_models.ipynb).)
You can define a problem most straightforwardly through a problem declaration
that gives the problem's name and its list of components. Thus in Figure 14-3 we have:

```ampl
problem Cutting_Opt: Cut, Number, Fill;
```

A new problem named Cutting_Opt is defined, and is specified to contain all of the
Cut variables, the `objective` Number, and all of the Fill constraints from the `model` in
Figure 14-2. At the same time, Cutting_Opt becomes the current problem. Any fixed
Cut variables are unfixed, while all other declared variables are fixed at their current values.
The `objective` Number is restored if it had been previously dropped, while all other
declared objectives are dropped; and similarly any dropped Fill constraints are
restored, while all other declared constraints are dropped.

For more complex models, the list of a problem's components typically includes several
collections of variables and constraints, as in this example from stoch1.run (one
of the examples from the AMPL web site):

```ampl
problem Sub: Make, Inv, Sell,
       Stage2_Profit, Time, Balance2, Balance;
```

By specifying an indexing-expression after the problem name, you can define an indexed
collection of problems, such as these in multi2.run (another web site example):

```ampl
problem SubII {p in PROD}: Reduced_Cost[p],
       {i in ORIG, j in DEST} Trans[i,j,p],
       {i in ORIG} Supply[i,p], {j in DEST} Demand[j,p];
```

For each `p` in the set PROD, a problem SubII[p] is defined. Its components include
the `objective` Reduced_Cost[p], the variables Trans[i,j,p] for each combination
of `i` in ORIG and `j` in DEST, and the constraints Supply[i,p] and
Demand[j,p] for each `i` in ORIG and each `j` in DEST, respectively.
A problem declaration's form and interpretation naturally resemble those of other
AMPL statements that specify lists of `model` components. The declaration begins with the
keyword problem, a problem name not previously used for any other `model` component,
an optional indexing expression (to define an indexed collection of problems), and a
colon. Following the colon is the comma-separated list of variables, objectives and constraints
to be included in the problem. This list may contain items of any of the following
forms, where "component" refers to any variable, `objective` or constraint:

- A component name, such as Cut or Fill, which refers to all components
having that name.
- A subscripted component name, such as Reduced_Cost[p], which
refers to that component alone.
- An indexing expression followed by a subscripted component name, such
as {i in ORIG} Supply[i,p], which refers to one component for each
member of the indexing set.

To save the trouble of repeating an indexing expression when several components are
indexed in the same way, the problem statement also allows an indexing expression followed
by a parenthesized list of components. Thus for example the following would be
equivalent:

```ampl
{i in ORIG} Supply1[i,p], {i in ORIG} Supply2[i,p],
{i in ORIG, j in DEST} Trans[i,j,p],
{i in ORIG, j in DEST} Use[i,j,p]
{i in ORIG} (Supply1[i,p], Supply2[i,p],
       {j in DEST} (Trans[i,j,p], Use[i,j,p]))
```

As these examples show, the list inside the parentheses may contain any item that is valid
in a component list, even an indexing expression followed by another parenthesized list.
This sort of recursion is also found in AMPL's print command, but is more general
than the list format allowed in `display` commands.

Whenever a variable, `objective` or constraint is declared, it is automatically added to
the current problem (or all current problems, if the most recent problem statement specified
an indexed collection of problems). Thus in our cutting-stock example, all of Figure 14-2's
`model` components are first placed by default into the problem Initial; then,
when the script of Figure 14-3 is run, the components are divided into the problems
Cutting_Opt and Pattern_Gen by use of problem statements. As an alternative,
we can declare empty problems and then fill in their members through AMPL declarations.
Figure 14-6 (cut2.mod) shows how this would be done for the Figure 14-2 models.
This approach is sometimes clearer or easier for simpler applications.

Any use of `drop`/`restore` or `fix`/`unfix` also modifies the current problem. The
`drop` command has the effect of removing constraints or objectives from the current
problem, while the `restore` command has the effect of adding constraints or objectives.
Similarly, the `fix` command removes variables from the current problem and the `unfix`

```ampl
problem Cutting_Opt;
param nPAT integer >= 0, default 0;
param roll_width;
set PATTERNS = 1..nPAT;
set WIDTHS;
param orders {WIDTHS} > 0;
param nbr {WIDTHS,PATTERNS} integer >= 0;
       check {j in PATTERNS}:
       sum {i in WIDTHS} i * nbr[i,j] <= roll_width;
var Cut {PATTERNS} >= 0;
minimize Number: sum {j in PATTERNS} Cut[j];
subject to Fill {i in WIDTHS}:
       sum {j in PATTERNS} nbr[i,j] * Cut[j] >= orders[i];
problem Pattern_Gen;
param price {WIDTHS};
var Use {WIDTHS} integer >= 0;
minimize Reduced_Cost:
       1 - sum {i in WIDTHS} price[i] * Use[i];
subject to Width_Limit:
       sum {i in WIDTHS} i * Use[i] <= roll_width;
```

**Figure 14-6:** Alternate definition of named cutting-stock problems (cut2.mod).

command adds variables. As an example, multi1.run uses the following problem
statements:

```ampl
problem MasterI: Artificial, Weight, Excess, Multi, Convex;
problem SubI: Artif_Reduced_Cost, Trans, Supply, Demand;
problem MasterII: Total_Cost, Weight, Multi, Convex;
problem SubII: Reduced_Cost, Trans, Supply, Demand;
```

to define named problems for phases I and II of its decomposition procedure. By contrast,
multi1a.run specifies

```ampl
problem Master: Artificial, Weight, Excess, Multi, Convex;
problem Sub: Artif_Reduced_Cost, Trans, Supply, Demand;
```

to define the problems initially, and then

```ampl
problem              Master;
       drop              Artificial; restore Total_Cost; fix Excess;
problem              Sub;
       drop              Artif_Reduced_Cost; restore Reduced_Cost;
```

when the time comes to convert the problems to a form appropriate for the second phase.
Since the names Master and Sub are used throughout the procedure, one loop in the
script suffices to implement both phases.
Alternatively, a `redeclare` problem statement can give a new definition for a
problem. The `drop`, `restore`, and `fix` commands above could be replaced, for
instance, by

```ampl
redeclare problem Master: Total_Cost, Weight, Multi, Convex;
redeclare problem Sub: Reduced_Cost, Trans, Supply, Demand;
```

Like other declarations, however, this cannot be used within a compound statement (if,
for or repeat) and so cannot be used in the multi1a.run example.

A form of the `reset` command lets you undo any changes made to the definition of a
problem. For example,

```ampl
reset problem Cutting_Opt;
```

resets the definition of Cutting_Opt to the list of components in the problem statement
that most recently defined it.

**Using named problems**

We next describe alternatives for changing the current problem. Any change will in
general cause different objectives and constraints to be dropped, and different variables to
be fixed, with the result that a different optimization problem is generated for the solver.
The values associated with `model` components are not affected simply by a change in the
current problem, however. All previously declared components are accessible regardless
of the current problem, and they keep the same values unless they are explicitly changed
by `let` or `data` statements, or by a `solve` in the case of variable and `objective` values
and related quantities (such as dual values, slacks, and reduced costs).

Any problem statement that refers to only one problem (not an indexed collection
of problems) has the effect of making that problem current. As an example, at the beginning
of the cutting-stock script we want to make first one and then the other named problem
current, so that we can adjust certain options in the environments of the problems.
The problem statements in cut1.run (Figure 14-3):

```ampl
problem Cutting_Opt: Cut, Number, Fill;
       option relax_integrality 1;
problem Pattern_Gen: Use, Reduced_Cost, Width_Limit;
       option relax_integrality 0;
```

serve both to define the new problems and to make those problems current. The analogous
statements in cut2.run are simpler:

```ampl
problem Cutting_Opt;
       option relax_integrality 1;
problem Pattern_Gen;
       option relax_integrality 0;
```

These statements serve only to make the named problems current, because the problems
have already been defined by problem statements in cut2.mod (Figure 14-6).
A problem statement may also refer to an indexed collection of problems, as in the
multi2.run example cited previously:

```ampl
problem SubII {p in PROD}: Reduced_Cost[p], ...
```

This form defines potentially many problems, one for each member of the set PROD.
Subsequent problem statements can make members of a collection current one at a time,
as in a loop having the form

```ampl
for {p in PROD} {
       problem SubII[p];
       ...
}
```

or in a statement such as problem SubII["coils"] that refers to a particular member.

As seen in previous examples, the `solve` statement can also include a problem name,
in which case the named problem is made current and then sent to the solver. The effect
of a statement such as `solve` Pattern_Gen is thus exactly the same as the effect of
problem Pattern_Gen followed by `solve`.

**Displaying named problems**

The command consisting of problem alone tells which problem is current:

```ampl
ampl: model cut.mod;
ampl: data cut.dat;
ampl: problem;
problem Initial;
ampl: problem Cutting_Opt: Cut, Number, Fill;
ampl: problem Pattern_Gen: Use, Reduced_Cost, Width_Limit;
ampl: problem;
problem Pattern_Gen;
```

The current problem is always Initial until other named problems have been defined.

The show command can give a list of the named problems that have been defined:

```ampl
ampl: show problems;
problems:   Cutting_Opt           Pattern_Gen
```

We can also use show to see the variables, objectives and constraints that make up a particular
problem or indexed collection of problems:

```ampl
ampl: show Cutting_Opt, Pattern_Gen;
problem Cutting_Opt: Fill, Number, Cut;
problem Pattern_Gen: Width_Limit, Reduced_Cost, Use;
```

and use expand to see the explicit objectives and constraints of the current problem,
after all `data` values have been substituted:

```ampl
ampl: expand Pattern_Gen;
minimize Reduced_Cost:
              -0.166667*Use[20] - 0.416667*Use[45] - 0.5*Use[50]
              - 0.5*Use[55] - 0.833333*Use[75] + 1;
subject to Width_Limit:
              20*Use[20] + 45*Use[45] + 50*Use[50] + 55*Use[55] +
              75*Use[75] <= 110;
```

See [Section 12.6](../12/12_6_accessing_model_and_solver_status.ipynb) for further discussion of show and expand.

**Defining and using named environments**

In the same way that there is a current problem at any point in an AMPL session, there
is also a current environment. Whereas a problem is a list of non-fixed variables and
non-dropped objectives and constraints, an environment records the values of all AMPL
options. By naming different environments, a script can easily switch between different
collections of option settings.

In the default mode of operation, which is sufficient for many purposes, the current
environment always has the same name as the current problem. At the start of an AMPL
session the current environment is named Initial, and each subsequent problem
statement that defines a new named problem also defines a new environment having the
same name as the problem. An environment initially inherits all the option settings that
existed when it was created, but it retains new settings that are made while it is current.
Any problem or `solve` statement that changes the current problem also switches to the
correspondingly named environment, with options set accordingly.

As an example, our script for the cutting stock problem (Figure 14-3) sets up the
`model` and `data` and then proceeds as follows:

```ampl
option solver cplex, solution_round 6;
option display_1col 0, display_transpose -10;
problem Cutting_Opt: Cut, Number, Fill;
option relax_integrality 1;
problem Pattern_Gen: Use, Reduced_Cost, Width_Limit;
option relax_integrality 0;
```

Options solver and three others are changed (by the first two option statements)
before any of the problem statements; hence their new settings are inherited by subsequently
defined environments and are the same throughout the rest of the script. Next a
problem statement defines a new problem and a new environment named
Cutting_Opt, and makes them current. The ensuing option statement changes
`relax_integrality` to 1. Thereafter, when Cutting_Opt is the current problem
(and environment) in the script, `relax_integrality` will have the value 1. Finally,
another problem and option statement do much the same for problem (and environment)
Pattern_Gen, except that `relax_integrality` is set back to 0 in that environment.
The result of these initial statements is to guarantee a proper setup for each of the subsequent
`solve` statements in the repeat loop. The result of `solve` Cutting_Opt is to
set the current environment to Cutting_Opt, thereby setting `relax_integrality`
to 1 and causing the linear relaxation of the cutting optimization problem to be solved.
Similarly the result of `solve` Pattern_Gen is to cause the pattern generation problem
to be solved as an `integer` program. We could instead have used option statements
within the loop to switch the setting of `relax_integrality`, but with this approach
we have kept the loop — the key part of the script — as simple as possible.

In more complex situations, you can declare named environments independently of
named problems, by use of a statement that consists of the keyword environ followed
by a name:

```ampl
environ Master;
```

Environments have their own name space. If the name has not been used previously as
an environment name, it is defined as one and is associated with all of the current option
values. Otherwise, the statement has the effect of making that environment (and its associated
option values) current.

A previously declared environment may also be associated with the declaration of a
new named problem, by placing environ and the environment name before the colon in
the problem statement:

```ampl
problem MasterII environ Master: ...
```

The named environment is then automatically made current whenever the associated
problem becomes current. The usual creation of an environment having the same name
as the problem is overridden in this case.

An indexed collection of environments may be declared in an environ statement by
placing an AMPL indexing expression after the environment name. The name is then
"subscripted" in the usual way to refer to individual environments.

Named environments handle changes in the same way as named problems. If an
option's value is changed while some particular environment is current, the new value is
recorded and is the value that will be reinstated whenever that environment is made current
again.

**Bibliography**

Vasˇek Chva´ tal, Linear Programming. Freeman (New York, NY, 1983). A general introduction to
linear programming that has chapters on the cutting-stock problem sketched in [Section 14.4](./14_4_named_problems.ipynb) and on
the Dantzig-Wolfe decomposition procedure that is behind the multi examples cited in [Section 14.5](#missing) <!--- xTODO missing reference --->.

Marshall L. Fisher, "An Applications Oriented Guide to Lagrangian Relaxation." Interfaces 15, 2
(1985) pp. 10-21. An introduction to the Lagrangian relaxation procedures underlying the trnloc2
script of [Section 14.3](./14_3_sending_suffixes_to_solvers.ipynb).
Robert Fourer and David M. Gay, "Experience with a Primal Presolve Algorithm." In Large
Scale Optimization: State of the Art, W. W. Hager, D. W. Hearn and P. M. Pardalos, eds., Kluwer
Academic Publishers (Dordrecht, The Netherlands, 1994) pp. 135-154. A detailed description of
the presolve procedure sketched in [Section 14.1](./14_1_presolve.ipynb).

Robert W. Haessler, "Selection and Design of Heuristic Procedures for Solving Roll Trim Problems
." Management Science 34, 12 (1988) pp. 1460-1471. Descriptions of real cutting-stock
problems, some amenable to the techniques of [Section 14.4](./14_4_named_problems.ipynb) and some not.

Leon S. Lasdon, Optimization Theory for Large Systems. Macmillan (New York, NY, 1970),
reprinted by Dover (Mineola, NY, 2002). A classic source for several of the decomposition and
generation procedures behind the scripts.